In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

CATALOG = "bucharest_mobility"
SILVER = "silver"
GOLD = "gold"

REALTIME_TABLE = f"{CATALOG}.{SILVER}.stop_time_updates"
STATIC_STOP_TIMES = f"{CATALOG}.{SILVER}.stop_times"


TARGET_TABLE = f"{CATALOG}.{GOLD}.stop_delay_observations"

In [0]:
realtime = (
    spark.table(REALTIME_TABLE)
    .filter(F.col("arrival_time").isNotNull())
)

static_stop_times = spark.table(STATIC_STOP_TIMES)

static_snapshots = (
    static_stop_times
    .select(
        "_snapshot_id",
        "_ingested_at",
    )
    .distinct()
)



In [0]:
realtime_with_static = (
    realtime.alias("rt")
    .join(
        static_snapshots.alias("ss"),
        F.col("ss._ingested_at")
        <= F.col("rt.ingested_at"),
        "left",
    )
)

window = (
    Window
    .partitionBy(
        "rt.snapshot_hash",
        "rt.entity_id",
        "rt.stop_sequence",
    )
    .orderBy(F.col("ss._ingested_at").desc())
)

realtime_with_static = (
    realtime_with_static
    .withColumn(
        "_static_rank",
        F.row_number().over(window),
    )
    .filter(F.col("_static_rank") == 1)
    .drop("_static_rank")
)

In [0]:
joined = (
    realtime_with_static.alias("rt")
    .join(
        static_stop_times.alias("st"),
        (
            F.col("rt._snapshot_id")
            == F.col("st._snapshot_id")
        )
        & (
            F.col("rt.trip_id")
            == F.col("st.trip_id")
        )
        & (
            F.col("rt.stop_sequence")
            == F.col("st.stop_sequence")
        ),
        "inner",
    )
    .select(
        F.col("rt.snapshot_hash"),
        F.col("rt.ingested_at"),
        F.col("rt.feed_timestamp"),
        F.col("rt.entity_id"),
        F.col("rt.trip_id"),
        F.col("rt.route_id"),
        F.col("rt.stop_id"),
        F.col("rt.stop_sequence"),

        F.col("rt.arrival_time").alias(
            "predicted_arrival"
        ),

        F.col("rt._snapshot_id").alias(
            "static_snapshot_id"
        ),

        F.col("st.arrival_time").alias(
            "scheduled_arrival_gtfs"
        ),

        F.col("st.arrival_seconds"),
    )
)

print(f"Matched observations: {joined.count():,}")

In [0]:
trip_updates = (
    spark.table(
        f"{CATALOG}.{SILVER}.trip_updates"
    )
    .select(
        "snapshot_hash",
        "entity_id",
        "start_date",
    )
)

joined = (
    joined.alias("j")
    .join(
        trip_updates.alias("tu"),
        (
            F.col("j.snapshot_hash")
            == F.col("tu.snapshot_hash")
        )
        & (
            F.col("j.entity_id")
            == F.col("tu.entity_id")
        ),
        "left",
    )
    .select(
        "j.*",
        F.col("tu.start_date"),
    )
)

In [0]:
gold_df = (
    joined
    .withColumn(
        "service_date",
        F.to_date(
            F.col("start_date"),
            "yyyyMMdd",
        ),
    )
    .withColumn(
        "scheduled_arrival_local",
        F.expr(
            """
            timestamp(service_date)
            + make_interval(
                0, 0, 0, 0, 0, 0,
                arrival_seconds
            )
            """
        ),
    )
    .withColumn(
        "scheduled_arrival",
        F.to_utc_timestamp(
            F.col("scheduled_arrival_local"),
            "Europe/Bucharest",
        ),
    )
    .withColumn(
        "predicted_delay_seconds",
        (
            F.col("predicted_arrival").cast("long")
            - F.col("scheduled_arrival").cast("long")
        ),
    )
)

In [0]:
gold_df = gold_df.select(
    "snapshot_hash",
    "ingested_at",
    "feed_timestamp",

    "entity_id",
    "trip_id",
    "route_id",
    "stop_id",
    "stop_sequence",

    "service_date",
    "scheduled_arrival_gtfs",
    "scheduled_arrival_local",
    "scheduled_arrival",
    "predicted_arrival",
    "predicted_delay_seconds",

    "static_snapshot_id",
)

In [0]:
display(
    gold_df.select(
        F.count("*").alias("observations"),

        F.round(
            F.avg("predicted_delay_seconds"),
            2,
        ).alias("avg_delay_seconds"),

        F.expr(
            "percentile_approx(predicted_delay_seconds, 0.5)"
        ).alias("median_delay_seconds"),

        F.min("predicted_delay_seconds").alias(
            "min_delay_seconds"
        ),

        F.max("predicted_delay_seconds").alias(
            "max_delay_seconds"
        ),
    )
)

In [0]:
if spark.catalog.tableExists(TARGET_TABLE):
    existing_snapshots = (
        spark.table(TARGET_TABLE)
        .select("snapshot_hash")
        .distinct()
    )

    new_gold_df = gold_df.join(
        existing_snapshots,
        on="snapshot_hash",
        how="left_anti",
    )
else:
    new_gold_df = gold_df

new_rows = new_gold_df.count()

if new_rows == 0:
    print(
        "SKIP stop_delay_observations: "
        "no new realtime snapshots"
    )
else:
    (
        new_gold_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(TARGET_TABLE)
    )

    print(
        f"LOADED stop_delay_observations: "
        f"{new_rows:,} rows"
    )

In [0]:
gold = spark.table(TARGET_TABLE)

duplicates = (
    gold
    .groupBy(
        "snapshot_hash",
        "entity_id",
        "stop_sequence",
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicates == 0, (
    f"stop_delay_observations: "
    f"{duplicates} duplicate grains"
)

print("✓ stop_delay_observations: grain unique")

In [0]:
null_delays = (
    gold
    .filter(
        F.col("predicted_delay_seconds").isNull()
    )
    .count()
)

null_schedule = (
    gold
    .filter(
        F.col("scheduled_arrival").isNull()
    )
    .count()
)

assert null_delays == 0
assert null_schedule == 0

print("✓ predicted delays populated")
print("✓ scheduled arrivals populated")
print(f"✓ total observations: {gold.count():,}")